# IMC hot pixel inspection & noise exclusion

Preparation and pixel accounting

**Author and pipeline lead:** Yang Cheng, University of Virginia (UVA), Christopher Deppmann and Melanie Worley laboratories.

**Acknowledgments:** Substantial technical assistance, mentoring, and materials from Eli Zunder’s laboratory.

**Development contributor:** ChatGPT (OpenAI), contributing to code, interface development, documentation, and tutorial construction.

This notebook supports Chapters 1–2 of the illustrated browser tutorial. It reads the exact supplied images, labels and population table. It does not choose a biological cutoff or apply corrections. Run in a Python environment with `numpy` and `tifffile`; install Jupyter separately if needed. Start from this notebook in the tutorial root folder.

The browser interface is the annotation tool: run `python serve.py` in a terminal, then open the printed address. Viewing an image does not mark a cell reviewed.


In [1]:
from pathlib import Path
import csv, json
import numpy as np
import tifffile

ROOT = Path.cwd()
assert (ROOT / "example/dataset.json").exists(), "Open Jupyter in the tutorial root folder."
config = json.loads((ROOT / "example/dataset.json").read_text())
print(config["dataset_name"])
print("Intensity provenance:", config["intensity_provenance"])
print("Mask semantics:", config["mask_semantics"])


Rat SCG IMC · 09082026 · five-ROI teaching subset
Intensity provenance: Steinbock 0.16.3 extraction with HPF=50; no spillover correction or normalization. Selected quantitative channels preserved exactly. Teaching subset, not independent biological replicates.
Mask semantics: User-validated whole-cell analysis regions with unchanged nuclear-origin labels. Original segmentation labels retained; neuronal assignments supplied separately.


## 1. Identify the paired input files

`rois.csv` pairs images and labels, provides explicit CYX/YX axes, and records X/Y calibration. `channels.csv` maps stable marker IDs to zero-based planes. `cell_metadata.csv` identifies selected neuronal analysis regions without removing other tissue from the masks. Keep these records with the images.


In [2]:
def read_rows(path):
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))

rois = read_rows(ROOT / "example/rois.csv")
channels = read_rows(ROOT / "example/channels.csv")
population = read_rows(ROOT / "example/cell_metadata.csv")
print("ROIs:", [r["roi"] for r in rois])
print("Channels:", len(channels), "(15 inspection markers + TH context)")
print("Selected cells:", sum(r["include"] == "true" for r in population))


ROIs: ['CoN_1_01', 'nLnK_1_04', 'nLwK_1_03', 'N_1_02', 'B_1_02']
Channels: 16 (15 inspection markers + TH context)
Selected cells: 2401


## 2. Select one marker and ROI

This teaching choice uses p-Akt S473 in N_1_02. The images are full-resolution quantitative values from Steinbock extraction with HPF=50; they are not display exports. The label image represents the original reviewed analysis regions. Cell identity is ROI plus positive mask label.


In [3]:
roi_name, marker_id = "N_1_02", "p_Akt_S473"
roi = next(r for r in rois if r["roi"] == roi_name)
channel = next(c for c in channels if c["marker_id"] == marker_id)
raw = tifffile.imread(ROOT / "example" / roi["image_path"])[int(channel["channel_index"])]
labels = tifffile.imread(ROOT / "example" / roi["mask_path"])
assert raw.shape == labels.shape
chosen_labels = [int(r["mask_label"]) for r in population
                 if r["roi"] == roi_name and r["include"] == "true"]
print("Image shape:", raw.shape, "selected cells:", len(chosen_labels))
print("Calibration:", roi["pixel_size_x_um"], "x", roi["pixel_size_y_um"], "µm")


Image shape: (1000, 1000) selected cells: 837
Calibration: 1 x 1 µm


## 3. Find a high-mean inspection candidate

The highest cell mean identifies something to inspect; it does not prove an artifact. Initial means include all original mask pixels, including zeros. Match the selected label in the browser to assess mask placement and spatial pattern.


In [4]:
# Recompute directly from the original pixels, independently of the preparation CSV.
cell_means = {lab: float(raw[labels == lab].mean(dtype=np.float64)) for lab in chosen_labels}
cell_label = max(cell_means, key=cell_means.get)
values = raw[labels == cell_label].astype(np.float64)
print("Candidate:", f"{roi_name}:{cell_label}")
print("Raw mean:", values.mean(), "maximum pixel:", values.max(), "pixels:", len(values))
area = len(values) * float(roi["pixel_size_x_um"]) * float(roi["pixel_size_y_um"])
print("Original analysis-region area:", area, "µm²")


Candidate: N_1_02:685
Raw mean: 1.67 maximum pixel: 30.5 pixels: 75
Original analysis-region area: 75.0 µm²


## 4. Bin pixels and reconcile their contributions

Illustrative boundaries 1, 3 and 5 produce zero, (0,1], (1,3], (3,5] and >5. This is a teaching display choice, not an accepted cutoff. Counts use the original cell area. Contributions to mean use the same full-cell denominator. Signal percentages use the full cell intensity sum.


In [5]:
edges = np.array([1., 3., 5.])
bin_index = np.where(values == 0, 0, np.searchsorted(edges, values, side="left") + 1)
names = ["I = 0", "0 < I <= 1", "1 < I <= 3", "3 < I <= 5", "I > 5"]
counts = np.bincount(bin_index, minlength=5)
sums = np.bincount(bin_index, weights=values, minlength=5)
print("Range | pixels | area % | contribution to mean | signal %")
for name, count, total in zip(names, counts, sums):
    signal_percent = 100 * total / values.sum() if values.sum() else float("nan")
    print(f"{name:12} | {count:4} | {100*count/len(values):8.3f} | {total/len(values):12.6f} | {signal_percent:8.3f}")
assert counts.sum() == len(values)
assert np.isclose(sums.sum(), values.sum())
assert np.isclose(sum(sums/len(values)), values.mean())


Range | pixels | area % | contribution to mean | signal %
I = 0        |   49 |   65.333 |     0.000000 |    0.000
0 < I <= 1   |   17 |   22.667 |     0.226667 |   13.573
1 < I <= 3   |    3 |    4.000 |     0.106667 |    6.387
3 < I <= 5   |    1 |    1.333 |     0.066667 |    3.992
I > 5        |    5 |    6.667 |     1.270000 |   76.048


## 5. Keep exploratory and accepted boundaries separate

Explore an intensity threshold, inspect the spatial pattern, then record the marker-specific decision and rationale in the browser. The early p-Akt prototype used I>5; the final All Marker workflow used I>=cutoff. Equality must be explicit. A value below the hot-pixel cutoff can still contain spread or background.


In [6]:
candidate = 5.0  # exploratory only
print("Pixels strictly > candidate:", int((values > candidate).sum()))
print("Pixels equal to candidate:", int((values == candidate).sum()))
print("Pixels >= candidate:", int((values >= candidate).sum()))
print("No pixels or cells have been removed in this notebook.")


Pixels strictly > candidate: 5
Pixels equal to candidate: 1
Pixels >= candidate: 6
No pixels or cells have been removed in this notebook.


## 6. Adapt the input preparation

Copy `templates/` to `my_inputs/`, replace the placeholder paths/channel identities/calibration, then run this command in the tutorial folder:

```bash
python code/prepare_dataset.py my_inputs/dataset.json --output results/my_review
```

To reproduce the bundled preparation, substitute `example/dataset.json` and a new output directory. The script refuses existing directories. Review `validation.json`, inspect image/mask alignment in the browser, and export all-marker review JSON to preserve decisions.

Aggregate detection, noise-spread masking, global/marker-specific exclusion and final retained-pixel measurements are later chapters. No biological classification is inferred by this notebook.
